# 03 — ML Attempt and Failure (the path not taken)

Before landing on the simple regime filter, we built a 77-factor LightGBM + Ridge-on-PCs ensemble. It achieved an OOS information coefficient of +0.55 on forward 20-day realised vol — a respectable result. But when used to time short-vol entries it actively *destroyed* P&L. This notebook shows the diagnostic that revealed the problem.

## 1. The signal looked good

In [ ]:
# After walk-forward 5-fold CV, our ensemble's OOS IC on y_rv_20f was +0.55
# Detail in the original `21_backtest_wrds.py` script in the research repo.
ic_lgb, ic_ridge, ic_ensemble = 0.46, 0.54, 0.55
print(f'LightGBM IC:    {ic_lgb:+.2f}')
print(f'Ridge-on-PC IC: {ic_ridge:+.2f}')
print(f'Ensemble IC:    {ic_ensemble:+.2f}')

## 2. But the strategy lost money

Three backtests on the same trade list:

* **(A) Always short, no signal**: the unfiltered VRP harvest. Sharpe **+0.45** net.
* **(B) Short only on bottom-decile edge_rank** (model says IV is most expensive vs predicted RV): Sharpe **−0.65** net.
* **(C) Always short *minus* spot check on Volmageddon and SPX-Feb-2016 days**: the daily P&L math reproduces the historical record correctly — i.e. the math isn't broken; the signal is.

In [ ]:
# These numbers come from the diagnostic backtest (script 22).
results = pd.DataFrame([
    {'strategy': 'always-short, no signal',         'sharpe_net': +0.45, 'ann_pnl': 4193, 'comment': 'baseline VRP'},
    {'strategy': 'edge_rank <= 0.10 (signal-gated)', 'sharpe_net': -0.65, 'ann_pnl': -3363, 'comment': 'signal anti-predictive'},
])
import pandas as pd
results

## 3. Why the signal anti-predicts

The `edge = predicted_RV - current_IV` signal is most negative when current IV has spiked relative to the model's slow-moving prediction. But that's *exactly the worst time to short vol* — IV is high for a reason and forward realised vol is also elevated.

The ML correctly identified high-IV days. It then incorrectly classified them as 'rich, sell' when actually they should be classified as 'dangerous, sit out'.

The lesson: for variance-risk harvesting, **the edge is in regime selection, not in point prediction**.

## 4. The pivot

With hindsight, the right use of features that *correlate with high-vol regimes* is to use them as **skip filters**, not as **edge signals**. That's what the next notebook builds.